In [1]:
import sys
import os
import pandas as pd
import numpy as np

In [1]:
import csv
import re
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import statsmodels.api as sm
import linearmodels as lm
import imgkit

ModuleNotFoundError: No module named 'imgkit'

In [3]:
from linearmodels import PanelOLS, RandomEffects
from scipy import stats
from linearmodels import RandomEffects
import statsmodels.api as sm
from linearmodels.panel import compare

In [4]:
final_df = pd.read_csv("data/prepared_data_for_regression.csv")
print(final_df.head(5))

      country  year  incidence  mortality  ln(GDP_per_capita)       bmi  \
0        fiji  2000  39.310026  33.722967            9.151239  0.303726   
1    cambodia  2000  11.807551  10.664151            7.561138  0.100004   
2    kiribati  2000  13.866566  12.601468            7.816584  0.307055   
3  kazakhstan  2000  35.284421  18.655098            9.467759  0.291659   
4     jamaica  2000  50.404926  25.599975            9.160964  0.240683   

     pop_65  urbanization_rate  fertility  female_labor_rate  \
0  3.437768             47.908      2.992             37.841   
1  2.816518             18.586      3.794             77.834   
2  3.413156             42.958      4.071                NaN   
3  6.693366             56.098      1.898             65.381   
4  6.060642             51.814      2.345             58.147   

   internet_penetration  health_exp  female_smoking  hosp_beds       MIR  
0              1.496850    3.424412            15.9       2.05  0.857872  
1             

In [5]:
initial_countries=final_df['country'].nunique()
print(initial_countries)

166


# OECD  Countries:

In [6]:
oecd_countries = ['austria', 'australia', 'belgium', 'canada', 'chile', 
                  'colombia', 'czech', 'denmark', 'estonia', 'finland', 
                  'france', 'germany', 'greece', 'hungary', 'iceland', 'ireland', 
                  'israel', 'italy', 'japan', 'korea', 'latvia', 'lithuania', 'luxembourg', 
                  'mexico', 'netherlands', 'new zealand', 'norway', 'poland', 'portugal', 'slovakia', 
                  'slovenia', 'spain', 'sweden', 'switzerland', 'turkiye', 'united kingdom', 'united states', 'costa rica']
print(len(oecd_countries))

38


# Calculate MIR (mortality to incidence rate):

In [7]:
MIR = final_df['mortality'] / final_df['incidence']
print(MIR)

0       0.857872
1       0.903164
2       0.908766
3       0.528706
4       0.507886
          ...   
3663    0.144413
3664    0.135142
3665    0.285674
3666    0.132557
3667    0.463373
Length: 3668, dtype: float64


# Create Dummy Variables:

In [8]:
final_df['is_oecd'] = final_df['country'].isin(oecd_countries).astype(int)

# create interaction :

In [9]:
final_df['bmi_x_oecd'] = final_df['bmi']* final_df['is_oecd']
print(final_df['is_oecd'].value_counts())

0    2884
1     784
Name: is_oecd, dtype: int64


# Dummy Variables: 

In [10]:
final_df['is_oecd']= final_df['country'].str.lower().str.strip().isin(oecd_countries).astype(int)
print(final_df['is_oecd'].value_counts())

0    2884
1     784
Name: is_oecd, dtype: int64


1. Aging Society Dummy: Based on WHO standards, societies with >10% population over 65 are considered "ageing" or "aged". This captures potential non-linear increases in cancer incidence due to population structure.

In [11]:
final_df['dm_high_aging_society'] = (final_df['pop_65'] > 10).astype(int)
print(final_df['dm_high_aging_society'].value_counts())

0    2457
1    1211
Name: dm_high_aging_society, dtype: int64


2. High Life Expectancy Dummy threshold of 75 years represents advanced healthcare systems and higher probability of disease detection.

In [12]:
print(final_df[['year', 'country', 'is_oecd', 'dm_high_aging_society']].head(100))

    year       country  is_oecd  dm_high_aging_society
0   2000          fiji        0                      0
1   2000      cambodia        0                      0
2   2000      kiribati        0                      0
3   2000    kazakhstan        0                      0
4   2000       jamaica        0                      0
..   ...           ...      ...                    ...
95  2000       denmark        1                      1
96  2001          oman        0                      0
97  2000       lebanon        0                      0
98  2000       namibia        0                      0
99  2000  saudi arabia        0                      0

[100 rows x 4 columns]


# interaction of is_oecd variable and GDP per capita per capita per capita per capita

In [13]:
final_df['log_gdp_is_oecd']= final_df['ln(GDP_per_capita)']*final_df['is_oecd']
print(final_df['log_gdp_is_oecd'].head(5))

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: log_gdp_is_oecd, dtype: float64


In [14]:
final_df['is_oecd'] = 0
final_df.loc[final_df['country'].isin(oecd_countries), 'is_oecd'] = 1

# interaction of is_oecd variable and gdp
final_df['log_gdp_is_oecd'] = final_df['ln(GDP_per_capita)'] * final_df['is_oecd']
# Filling NaNs in Interaction Term: Since 'NaN * 0 = NaN' in Python, rows where is_oecd is 0 but ln(GDP per capita)  is missing(NAN) 
# would incorrectly stay as NaN (and be counted in the results), then filled these with 0 to ensure only real OECD countries with data are counted.
final_df['log_gdp_is_oecd'] = final_df['log_gdp_is_oecd'].fillna(0)

# counting contries :
oecd_count = final_df[final_df['is_oecd'] == 1]['country'].nunique()
interaction_countries = final_df[final_df['log_gdp_is_oecd'] != 0]['country'].nunique()

In [15]:
print(final_df[['country','ln(GDP_per_capita)','is_oecd','log_gdp_is_oecd','dm_high_aging_society']].sample(20).round(2))

                country  ln(GDP_per_capita)  is_oecd  log_gdp_is_oecd  \
956            cambodia                8.10        0             0.00   
2648             poland               10.51        1            10.51   
1157            denmark               11.05        1            11.05   
3434               mali                7.96        0             0.00   
2961           cambodia                8.72        0             0.00   
2921            lebanon                9.66        0             0.00   
1662       turkmenistan                8.98        0             0.00   
1809             zambia                8.11        0             0.00   
3087               oman               10.53        0             0.00   
2313             serbia                9.85        0             0.00   
917              norway               11.34        1            11.34   
716                iraq                9.22        0             0.00   
3181              malta               10.96        

# delete percentage of rows with missing value(NA):

In [16]:
# report of missing values
missing_report = (final_df.isnull().sum() / len(final_df) * 100).round(2)
print(missing_report[missing_report > 0])

ln(GDP_per_capita)       1.09
female_labor_rate        2.48
internet_penetration     1.64
female_smoking          10.88
hosp_beds                0.63
dtype: float64


In [17]:
clean_df= final_df.dropna(subset=['MIR', 'ln(GDP_per_capita)', 'pop_65', 'urbanization_rate', 'fertility', 'female_labor_rate', 'log_gdp_is_oecd', 'internet_penetration', 'health_exp', 'female_smoking','hosp_beds'])
print(f" Missing values after deleting NA in clean_df: {clean_df.isnull().sum()}")

 Missing values after deleting NA in clean_df: country                  0
year                     0
incidence                0
mortality                0
ln(GDP_per_capita)       0
bmi                      0
pop_65                   0
urbanization_rate        0
fertility                0
female_labor_rate        0
internet_penetration     0
health_exp               0
female_smoking           0
hosp_beds                0
MIR                      0
is_oecd                  0
bmi_x_oecd               0
dm_high_aging_society    0
log_gdp_is_oecd          0
dtype: int64


# Building stepwise regression

# Model 1: Adding Macro-Economic Variables:

In [18]:
df_step= clean_df.set_index(['country', 'year'])
exog_1_fe= sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate']])
model_1_fe=PanelOLS(df_step['MIR'],exog_1_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(model_1_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    MIR   R-squared:                        0.0955
Estimator:                   PanelOLS   R-squared (Between):              0.4282
No. Observations:                3124   R-squared (Within):               0.3262
Date:                Wed, Aug 19 2026   R-squared (Overall):              0.4218
Time:                        18:04:40   Log-likelihood                    4840.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      156.51
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(2,2963)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             10.695
                            

# Model 2: Adding Demographic Variables:

In [19]:
exog_2_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65','fertility','dm_high_aging_society']])
model_2_fe=PanelOLS(df_step['MIR'], exog_2_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(model_2_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    MIR   R-squared:                        0.3468
Estimator:                   PanelOLS   R-squared (Between):              0.2276
No. Observations:                3124   R-squared (Within):               0.2667
Date:                Wed, Aug 19 2026   R-squared (Overall):              0.2245
Time:                        18:04:40   Log-likelihood                    5349.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      314.26
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(5,2960)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             24.318
                            

# Model 3: Adding Lifestyle Variables:

In [20]:
exog_3_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65', 'dm_high_aging_society','fertility','bmi', 'female_smoking']])
model_3_fe=PanelOLS(df_step['MIR'], exog_3_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered', cluster_entity=True)
print(model_3_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    MIR   R-squared:                        0.3579
Estimator:                   PanelOLS   R-squared (Between):              0.3090
No. Observations:                3124   R-squared (Within):               0.2280
Date:                Wed, Aug 19 2026   R-squared (Overall):              0.3001
Time:                        18:04:41   Log-likelihood                    5376.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      235.58
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                  F(7,2958)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             18.807
                            

# Model 4: Adding Systemic and Digital variables:

In [21]:
exog_4_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65','fertility','dm_high_aging_society','female_smoking', 'bmi','log_gdp_is_oecd', 'female_labor_rate', 'internet_penetration', 'health_exp']])
model_4_fe=PanelOLS(df_step['MIR'], exog_4_fe, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(model_4_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    MIR   R-squared:                        0.4283
Estimator:                   PanelOLS   R-squared (Between):             -0.7880
No. Observations:                3124   R-squared (Within):               0.5703
Date:                Wed, Aug 19 2026   R-squared (Overall):             -0.7512
Time:                        18:04:41   Log-likelihood                    5557.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      201.17
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(11,2954)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             24.348
                            

# create female_labor_rate * is_oecd variable:

In [22]:
df_step['female_labor_rate * is_oecd']= df_step['female_labor_rate' \
'']* df_step['is_oecd']

# Model 5: Adding Interaction Terms Variable:

In [23]:
exog_5_fe=sm.add_constant(df_step[['ln(GDP_per_capita)', 'urbanization_rate','pop_65', 'dm_high_aging_society','fertility','bmi', 'female_smoking', 'female_labor_rate', 'internet_penetration', 
                                   'log_gdp_is_oecd','female_labor_rate * is_oecd']])
model_5_fe=PanelOLS(df_step['MIR'], exog_5_fe, entity_effects=True, time_effects= True).fit(cov_type='clustered',
                                                                                                  cluster_entity=True)
print(model_5_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    MIR   R-squared:                        0.4494
Estimator:                   PanelOLS   R-squared (Between):             -0.2175
No. Observations:                3124   R-squared (Within):               0.5184
Date:                Wed, Aug 19 2026   R-squared (Overall):             -0.1954
Time:                        18:04:41   Log-likelihood                    5616.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      219.19
Entities:                         136   P-value                           0.0000
Avg Obs:                       22.971   Distribution:                 F(11,2954)
Min Obs:                       17.000                                           
Max Obs:                       24.000   F-statistic (robust):             23.715
                            

# Comparing results of all Fixed Effect models:

In [24]:
Compare_results_fixed_Effect = compare({
    'FE model 1': model_1_fe,
    'FE model 2': model_2_fe,
    'FE model 3': model_3_fe,
    'FE model 4': model_4_fe,
    'FE model 5': model_5_fe}, stars=True)

html_table = Compare_results_fixed_Effect.summary.as_html()

# Clean all default R-squared rows at once
html_table = re.sub(r'<tr>\s*<th>R-[Ss]quared.*?</tr>', '', html_table, flags=re.DOTALL)

# Insert our custom row directly before F-statistic
r2_vals = "".join([f"<td>{m.rsquared_within:.6f}</td>" for m in [model_1_fe, model_2_fe, model_3_fe, model_4_fe, model_5_fe]])
html_table = html_table.replace('<tr>\n  <th>F-statistic', f'<tr>\n  <th>R-squared</th>\n{r2_vals}\n</tr>\n<tr>\n  <th>F-statistic')

# Truncate all numbers to exactly 3 decimals
html_table = re.sub(r'(\.\d{3})\d+', r'\1', html_table)

style = "<style> table.simpletable {border-top: 1px solid black; border-bottom: 1px solid black; border-collapse: collapse; font-family: 'Times New Roman';} table.simpletable td, table.simpletable th {border: none; padding: 5px 15px;} table.simpletable tr:first-child {border-bottom: 1px solid black;} </style>"
note = "<br><i>Note: Standard errors in parentheses. * p<0.1, ** p<0.05, *** p<0.01</i>"

with open('MIR_comparison_Fixed_Effect.html', 'w', encoding='utf-8') as f:
    f.write(style + html_table + note)

# CSV File for ML model:

In [25]:
# creating CSV file for ML model:
df_step.to_csv('preparing_MIR_data_for_ML.csv' , index= False)

In [26]:
with open('MIR_ols_r2-within.txt', 'w') as f:
    f.write(str(model_5_fe.rsquared_within))